# Get Signal Metrics in Chatlogs

This script applies the various construct signal measures specified in `signals_config.csv`
to the **PII-redacted** level-1 turn logs produced by `01_llm_prep_logs.ipynb` (e.g., `P1_log_level1.csv`,
`P2_log_level1.csv`, pulled from Box's `redacted_logs/` folder.
<br><br>
Each signal declares a `method` in
`signals_config.csv`, and `signal_methods` maps method names to
measurement functions. This allows us to easily swap out measurement methods (e.g., SBERT to
a lexicon) by adding a function and registering it in the file.
<br><br>
Before scoring, every turn is run once through a code-filter (`clean_and_check`) that strips code blocks and flags turns that are mostly code as unscorable (`is_code_like`), so pasted code doesn't get scored as if it were more meaningful natural language.
<br><br>

____________________________

*Notes:*
* *Currently, we measure the constructs **Attachment Anxiety** and **Cognitive Distortions**. Each construct has several signals.*

* *Four methods are actively run for each signal: **Contextualized Construct Representations (CCR)** (compares chatlog turns to ECR-R anchor items), a **lexicon**/keyword-matching method, a direct **SBERT** cosine-similarity method against hand-written prototype sentences, and a **zero-shot LLM prompting** (Phi-3.5-mini-instruct)*

* *More constructs may be added later as additional rows in
`signals_config.csv`*
<br><br>
____________________________
<br>

**Input and Output**

- `signals_config.csv` <= input — one row per signal: `construct`, `signal`, `method`,
  `params_key`, `active`
- `signal_params.json` <= input method-specific parameters for signals (anchor items for CCR, key_phrases for lexicon, prototype_sentences for SBERT, definition for zero-shot), keyed by `params_key`
- `redacted_logs/{participant}_log_level1.csv` <= input, one file per participant, downloaded from Box
- `metrics/{participant}_{signal_slug}_{method}.csv` <= output, one file per (participant, signal, method) combination, uploaded back to Box



# Get Signal Metrics in Chatlogs

This script applies the various construct signal measures specified in `signals_config.csv`
to the level-1 turn logs produced by the preprocessing notebook (e.g., `P1_log_level1.csv`,
`P2_log_level1.csv`).
<br><br>
Each signal declares a `method` in
`signals_config.csv`, and `signal_methods` maps method names to
measurement functions. This allows us to easily swap out measurement methods (e.g., LLM zero-shot annotation to
a lexicon) by adding a function and registering it in the file.
<br><br>

____________________________

*Notes:*
* *Currently, I measure the following constructs and signals*
  * **Attachment Anxiety**
    * Abandonment & Loss
Worry
    * Relational Imbalance
    * Relationship Self-doubt
    * Fear of Being Known
    * Anger at Unmet Needs
  * **Cognitive Distortions**
    * Ruminative Self-Focus
    * Future-Oriented Threat Language
    * Negative Global Self-Schema

* *I measure some of these signals using Contextualized Construct Representations (CCR), which compares chatlog messages  to ECR-R anchor items ([Fraley et al., 2000, Journal of Personality and Social Psychology](https://pubmed.ncbi.nlm.nih.gov/10707340/)).*

* *We may later add different measurement methods and additional constructs and signals (e.g. Cognitive Distortions, Zero-shot LLM Prompting)*

* *More constructs will be added later as additional rows in
`signals_config.csv`*
<br><br>
____________________________
<br>

**Input and Output**

- `signals_config.csv` <= input — one row per signal: `construct`, `signal`, `method`,
  `params_key`, `active`
- `signal_params.json` <= input method-specific parameters for signals (anchor items for CCR, but this may differ for different measurements such as lexicon terms, LLM prompt templates, thresholds, whatever a given method needs), keyed by `params_key`

## Part 0: Setup Environment

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

import os

# attempt to deal with Zero-shot method's GPU out-of-memory errors
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install pyccr --quiet
import sys
!{sys.executable} -m pip install "boxsdk==3.9.2" --quiet

from google.colab import userdata
from boxsdk import Client, OAuth2

access_token = userdata.get('BOX_DEVELOPER_TOKEN')
auth = OAuth2(client_id=None, client_secret=None, access_token=access_token)
client = Client(auth)

me = client.user().get()
print(f"Authenticated as: {me.name} ({me.login})")

In [ ]:
# Box folder/file IDs — from 01's output, plus new ones for this notebook's inputs/outputs
box_redacted_logs_folder_id = "401874734353"    # same folder used in 01's upload step
box_metrics_folder_id = "404285985307"           # new Box folder for this notebook's output
box_signals_config_file_id = "2360417963361"      # signals_config.csv in Box
box_signal_params_file_id = "2362064252761"       # signal_params.json in Box

root_file_path = Path("/content/logs")
level1_logs_dir = root_file_path / "level1_logs"   # local mirror of redacted logs
metrics_dir = root_file_path / "metrics"
config_dir = root_file_path
tmp_dir = root_file_path / "_ccr_tmp"

level1_logs_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)
tmp_dir.mkdir(parents=True, exist_ok=True)

signals_config_path = config_dir / "signals_config.csv"
signal_params_path = config_dir / "signal_params.json"

text_col = "message_content"
actor_col = "actor"
timestamp_col = "timestamp"
turn_id_col = "turn_id"
participant_col = "participant_id"
word_count_col = "word_count"

# Pull redacted logs (NOT raw/unredacted level1) into the local mirror
for item in client.folder(box_redacted_logs_folder_id).get_items():
    with open(level1_logs_dir / item.name, 'wb') as f:
        client.file(item.id).download_to(f)
    print(f"  downloaded {item.name}")

# Pull config files
with open(signals_config_path, 'wb') as f:
    client.file(box_signals_config_file_id).download_to(f)
with open(signal_params_path, 'wb') as f:
    client.file(box_signal_params_file_id).download_to(f)

## Part 1: Load Signals and  Method Parameters

### Load `signals_config.csv`

Edit this file directly to add constructs/signals or swap a signal's scoring method.

active=False rows are skipped by the main loop (for signals that are defined but not ready to run).

In [ ]:
if not signals_config_path.exists():
    raise FileNotFoundError(
        f"{signals_config_path} not found. Edit it manually to add/remove "
        f"signals or methods before running this notebook."
    )

signals_config = pd.read_csv(signals_config_path)
signals_config["active"] = signals_config["active"].astype(bool)
signals_config

### Load `signal_params.json`

Edit this file directly to update method-specific parameters (method is specified by params_key in signals_config.csv entries). Each entry is a dict where the contents depend on the method(s) that use it (e.g. anchor_items for CCR, key_phrases for lexicon, prototype_sentences for SBERT, prompts/examples for zero-shot LLM prompting).

In [ ]:
if not signal_params_path.exists():
    raise FileNotFoundError(
        f"{signal_params_path} not found. Edit it manually to add/update "
        f"anchor_items, key_phrases, or prototype_sentences before running "
        f"this notebook."
    )

with open(signal_params_path) as f:
    signal_params_by_key = json.load(f)

### Part 1.5: Methods to Filter Out Code-like Text

* Strip code from messages
  * (` ``` `) and (`code`) blocks
  * Syntax-like regex
  * Remaining text must be > 10 chars to have meaninful interpretations (CCR uses > 10 char threshold)



In [ ]:
code_fence_re = re.compile(r"```.*?```", re.DOTALL)
inline_code_re = re.compile(r"`[^`\n]+`")

# Syntax-anchored: requires actual code shape, not just a keyword appearing anywhere.
  # checks for syntax regex, symbol density, odd indentation
code_line_re = re.compile(
    r"^\s*("
    r"import\s+\w|from\s+\w+\s+import|"
    r"def\s+\w+\s*\(|class\s+\w+\s*[:\(]|"
    r"return\s|public\s+\w|static\s+\w|void\s+\w|"
    r"package\s+\w|throws\s+\w|System\.out|"
    r"\bfor\s*\(|\bwhile\s*\(|"
    r"var\s+\w+\s*=|const\s+\w+\s*=|let\s+\w+\s*=|"
    r"\}\s*$|\{\s*$"
    r")"
)

min_scorable_chars = 10 # Reponse may be too short to be meaningful


def _line_is_code_like(line: str) -> bool:
    stripped = line.strip()
    if not stripped:
        return False
    if code_line_re.match(stripped):
        return True
    symbol_chars = sum(c in "{}[]();=<>" for c in stripped)
    if symbol_chars / len(stripped) > 0.20:
        return True
    if line.startswith("    ") or line.startswith("\t"):
        return True
    return False


def clean_and_check(text) -> tuple:
    """
    Strip code content from a turn's text.

    Returns (is_code_like, cleaned_text). is_code_like=True means there isn't enough
    real language left to score (too short after stripping, or an unambiguous error
    traceback) — cleaned_text is '' in that case. Otherwise cleaned_text is what
    should actually get embedded and scored, not the raw turn text.

    """
    if not isinstance(text, str):
        return True, ""
    t = text.strip()
    if not t:
        return True, ""
    if any(x in t for x in ["Traceback", "Exception", "File \""]):
        return True, ""

    cleaned = code_fence_re.sub(" ", t)
    cleaned = inline_code_re.sub(" ", cleaned)

    kept_lines = [line for line in cleaned.split("\n") if not _line_is_code_like(line)]
    cleaned = " ".join(kept_lines)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    if len(cleaned) < min_scorable_chars:
        return True, ""
    return False, cleaned


## Part 2: Register Measurement Methods

`signal_methods` maps a `method` string (from `signals_config.csv`) to a scoring
function. Every method function is in the form:
```
score_fn(turns_df, signal_params, signal_name) -> pd.DataFrame
```

**Inputs**
* `turns_df`: the turn-level dataframe
* `signal_params`: a **dict** holding whatever's stored under this signal's `params_key` in `signal_params.json` (e.g. sentences for CCR)
* `signal name`: name of the signal being scored (e.g. Abandonment_Loss_Worry)

**Outputs**
* a dataframe **with the same index as `turns_df`, containing one
or more numeric score columns, whatever the method itself produces. (e.g. mean and max cosine similarity scores for CCR)

### 2.1 Contextualized Construct Representation (CCR)

Contextualized Construct Representations (CCR) is a method for detecting psychological constructs in text ([Atari et al., 2023, PsyArXiv](https://www.researchgate.net/profile/Mohammad-Atari-3/publication/368985024_Contextualized_Construct_Representation_Leveraging_Psychometric_Scales_to_Advance_Theory-Driven_Text_Analysis/links/6529a55006bdd619c48c11aa/Contextualized-Construct-Representation-Leveraging-Psychometric-Scales-to-Advance-Theory-Driven-Text-Analysis.pdf)).


It works by embedding a set of questionnaire items (in this case, the ECR-R items which measure insecure Attachment stypes) into the same SBERT embedding space as the conversational text (user-AI turns). It then computes cosine similarity between each turn and those items. For every turn, this yields a mean and a max cosine similarity across the questionnaire items belonging to that signal.


Because the number of ECR-R items varies by signal, the resulting mean and max similarities are noisier for signals with fewer items: each score is being averaged (or maxed) over a smaller sample, so a single unusually high or low similarity has a much bigger effect on the result.

In [ ]:
# # Scoring function for CCR
# from ccr import ccr_wrapper  # from the pyccr package
# import ccr.utils


# # Fix ccr bug in pyccr's own encode_column function having to do with handling
# # rows with missing text
# def _patched_encode_column(model, filename, col_name):
#     df = pd.read_csv(filename)
#     df = df.dropna(subset=[col_name])
#     df["embedding"] = list(model.encode(df[col_name].tolist()))
#     return df

# ccr.utils.encode_column = _patched_encode_column

# def compute_ccr_signal(turns_df: pd.DataFrame, signal_params: dict,
#                         signal_name: str) -> pd.DataFrame:
#     """
#     Expects turns_df to already have 'is_code_like' and 'cleaned_text' columns
#     (added once per participant in the main loop, not recomputed per signal).

#     Returns two columns (mean_score, max_score) because CCR aggregates similarity
#     across multiple anchor items per signal; this is a property of CCR specifically,
#     not a requirement other methods need to follow.
#     """

#     signal_anchor_items = signal_params.get("anchor_items", [])
#     if not signal_anchor_items:
#         raise ValueError(
#             f"No anchor items for '{signal_name}' — check signal_params.json's "
#             f"'anchor_items' key for this signal's params_key."
#         )

#     valid_mask = ~turns_df["is_code_like"]

#     # Initialize all output as NaN before scoring so non-scored turns are distinguishable
#     result_df = pd.DataFrame(
#         {"mean_score": np.nan, "max_score": np.nan}, index=turns_df.index
#     )

#     if not valid_mask.any():
#         return result_df

#     # Scoring uses cleaned_text (code stripped out), not the raw message_content
#     valid_df = pd.DataFrame({
#         turn_id_col: turns_df.loc[valid_mask, turn_id_col],
#         text_col: turns_df.loc[valid_mask, "cleaned_text"],
#     })

#     # ccr_wrapper (from pyccr) is a file-based API and expects CSV paths on disk:
#     # we use a scratch folder "tmp_dir" to deal with this
#     data_csv = tmp_dir / f"_valid_turns_{signal_name}.csv"
#     q_csv = tmp_dir / f"_questionnaire_{signal_name}.csv"
#     valid_df.to_csv(data_csv, index=False)
#     pd.DataFrame({"item": signal_anchor_items}).to_csv(q_csv, index=False)

#     # Positional args
#     result = ccr_wrapper(str(data_csv), text_col, str(q_csv), "item")

#     # Per-item similarity columns
#     passthrough_cols = set(valid_df.columns)
#     item_cols = [
#         c for c in result.columns
#         if c not in passthrough_cols and result[c].dtype.kind in "fi"
#     ]
#     if not item_cols:
#         raise ValueError(
#             f"No numeric similarity columns detected in ccr_wrapper output for "
#             f"'{signal_name}'. Columns found: {result.columns.tolist()} — "
#             f"check ccr_wrapper's output format (this was flagged as version-dependent before)."
#         )
#     if turn_id_col not in result.columns:
#         raise ValueError(
#             f"ccr_wrapper output for '{signal_name}' dropped '{turn_id_col}' — "
#             f"cannot align scores back to turns_df. Columns found: {result.columns.tolist()}"
#         )

#     # For signal being measured, compute MEAN cosine similarity and MAX cosine
#     # similarity of turn against all anchor items in the signal
#     result["mean_score"] = result[item_cols].mean(axis=1)
#     result["max_score"] = result[item_cols].max(axis=1)
#     scored = result.set_index(turn_id_col)[["mean_score", "max_score"]]

#     aligned = turns_df[[turn_id_col]].join(scored, on=turn_id_col)
#     result_df.loc[:, ["mean_score", "max_score"]] = aligned[["mean_score", "max_score"]].values
#     return result_df


### 2.2 Lexicon

A dictionary/keyword-matching method that compares a custom dictionary of key words/phrases for each signal to the user-AI turns. Dictionary-approaches are widely used in detecting psychological, emotional, and structural signals from text (LIWC: [Boyd et al., 2022, University of Texas at Austin](https://www.researchgate.net/profile/Ryan-Boyd-8/publication/358725479_The_Development_and_Psychometric_Properties_of_LIWC-22/links/6210f62c4be28e145ca1e60b/The-Development-and-Psychometric-Properties-of-LIWC-22.pdf); VADER: [Hutto & Gilbert, 2014, ICWSM](https://ojs.aaai.org/index.php/icwsm/article/view/14550))


For each turn, matches are counted and normalized by word count (matches per 1,000 words), giving a single score per turn.

In [ ]:
# # Scoring function for Lexicon (dictionary/keyword matching)

# def compute_lexicon_signal(turns_df: pd.DataFrame, signal_params: dict,
#                             signal_name: str) -> pd.DataFrame:
#     """
#     Counts occurrences of any phrase in signal_params['key_phrases'] within
#     each turn's cleaned_text, normalized to matches-per-1000-words (1000
#     is arbitrary, we could use 100) so short and long turns are comparable.

#     Unlike CCR, there's no natural multi-anchor aggregation here so we
#     return a single 'score' column instead of max/min scores.
#     """

#     key_phrases = signal_params.get("key_phrases", [])
#     if not key_phrases:
#         raise ValueError(
#             f"No key_phrases for '{signal_name}' — check signal_params.json's "
#             f"'key_phrases' key for this signal's params_key."
#         )

#     patterns = [re.compile(p, re.IGNORECASE) for p in key_phrases]

#     result_df = pd.DataFrame({"score": np.nan}, index=turns_df.index)

#     valid_mask = ~turns_df["is_code_like"]
#     if not valid_mask.any():
#         return result_df

#     def _score_text(text: str) -> float:
#         word_count = max(len(text.split()), 1)

#         # number of distinct patterns that matched at least once:
#         # a turn repeating one phrase five times doesn't score higher than a
#         # turn hitting five different phrases once each
#         raw_count = sum(1 for pat in patterns if pat.search(text))

#         # Normalize score: makes short and long turns comparable
#         # arbitrary per 1000 words, if wanted to do word percentage, could use * 100 instead
#         return (raw_count / word_count) * 1000

#     # Scores cleaned_text (code stripped out)
#     result_df.loc[valid_mask, "score"] = (
#         turns_df.loc[valid_mask, "cleaned_text"].apply(_score_text)
#     )
#     return result_df

### 2.3 SBERT

Uses the same SBERT embedding-similarity ([Reimers & Gurevych, 2019, EMNLP-IJCNLP](https://aclanthology.org/D19-1410.pdf)) approach as the CCR method above but compares each turn against created prototype sentences instead of the actual ECR-R items; comparable to CCR.


Each signal has a fixed set of 4 prototype sentences, so the mean and max cosine similarities are computed the same way as CCR, but are always averaged (or maxed) over exactly 4 items regardless of the signal, likely making them less noisy.

In [ ]:
# # Scoring function for SBERT direct (baseline semantic similarity)

# from sentence_transformers import SentenceTransformer, util

# # Loaded model once at import time and reused across all signals/participants:
# # same all-MiniLM-L6-v2 model as CCR method so results are comparable between
# # the two methods.
# _sbert_model = SentenceTransformer("all-MiniLM-L6-v2")


# def compute_sbert_signal(turns_df: pd.DataFrame, signal_params: dict,
#                           signal_name: str) -> pd.DataFrame:
#     """
#     Direct SBERT cosine-similarity baseline against signal_params['prototype_sentences']
#     mean_score/max_score are aggregated across the prototype set,
#     PER TURN
#     """

#     prototype_sentences = signal_params.get("prototype_sentences", [])
#     if not prototype_sentences:
#         raise ValueError(
#             f"No prototype_sentences for '{signal_name}' — check signal_params.json's "
#             f"'prototype_sentences' key for this signal's params_key."
#         )

#     valid_mask = ~turns_df["is_code_like"]

#     result_df = pd.DataFrame(
#         {"mean_score": np.nan, "max_score": np.nan}, index=turns_df.index
#     )

#     if not valid_mask.any():
#         return result_df

#     # Scores cleaned_text (code stripped out)
#     valid_texts = turns_df.loc[valid_mask, "cleaned_text"].tolist()

#     # Embed both protype sentences for signal AND turn (either user or
#     # AI message)

#     prototype_embeddings = _sbert_model.encode(prototype_sentences, convert_to_tensor=True)
#     turn_embeddings = _sbert_model.encode(valid_texts, convert_to_tensor=True)

#     # Turns x prototypes cosine similarity matrix
#     # Compute cosine similarity between the turn and every prototype sentence
#     # in the current signal.
#     sims = util.cos_sim(turn_embeddings, prototype_embeddings).cpu().numpy()

#     # mean cosine similarity of turn against all prototype sentences in the signal
#     result_df.loc[valid_mask, "mean_score"] = sims.mean(axis=1)
#     # max cosine similarity of turn against all prototype sentences in the signal
#     result_df.loc[valid_mask, "max_score"] = sims.max(axis=1)

#     return result_df

### 2.4 LLM Zero-shot
We use Phi-3.5-mini-instruct (lightweight local model since it's free) but the model can be swapped out later if we want to pay for an API key. This model was used and validated in ([Jia et al., 2025](https://arxiv.org/pdf/2507.08031)).

For each user or AI turn, we prompt the model with a signal's definition and the turn's text. Then we give it a binary question "does this turn clearly express the construct described?" By using a more probabilistic prompting, we get the model's output probability assigned to the answer "Yes" versus "No"  This gives us a continuous 0.0–1.0 score (the model's confidence that the construct is present).

*Notes*
* *Chose this probablistic binary question over a free answer question (asking the model to give an 0.0-1.0 score) because the former approach often gave discrete extreme scores in our data and in ([Li et al., 2025](https://arxiv.org/pdf/2501.15453)). Li et al., 2025 also used this probabilistic approach as a follow up in an attempt to resolve the former method's issue.*
* *Turns are scored in batches (of size 8) rather than individually since it takes too much compute on my computer*

In [ ]:
# Scoring function for LLM Zero-shot (local model, no API cost)

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import torch.nn.functional as F
import gc

# Lightweight open language model with Transformer architecture released by Microsoft
# free to use from Huggingface
_ZEROSHOT_MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"
_zs_tokenizer = None
_zs_model = None


# Load model
def _load_zeroshot_model():
    global _zs_tokenizer, _zs_model
    if _zs_model is None:
        _zs_tokenizer = AutoTokenizer.from_pretrained(_ZEROSHOT_MODEL_NAME)
        # messages in batch have different lengths so put padding tokens on left
        _zs_tokenizer.padding_side = "left"
        if _zs_tokenizer.pad_token is None:
            _zs_tokenizer.pad_token = _zs_tokenizer.eos_token

        _zs_model = AutoModelForCausalLM.from_pretrained(
            _ZEROSHOT_MODEL_NAME,
            dtype=torch.float16 if torch.cuda.is_available() else torch.float32, # try to save space
            device_map={"": 0} if torch.cuda.is_available() else None,
        )
        _zs_model.eval()
    return _zs_tokenizer, _zs_model


def _yes_no_token_ids(tokenizer):
    """
    Collects token IDs for case variants of Yes/No, since models don't always
    emit the exact casing prompted for. Each variant is checked to confirm it
    encodes to a single token before being included -- multi-token variants
    are skipped rather than silently mis-scored.
    """
    yes_variants = ["Yes", "yes", "YES"]
    no_variants = ["No", "no", "NO"]

    def _single_token_ids(variants):
        ids = []
        for v in variants:
            encoded = tokenizer.encode(v, add_special_tokens=False)
            if len(encoded) == 1:
                ids.append(encoded[0])
        return ids

    return _single_token_ids(yes_variants), _single_token_ids(no_variants)

# What to ask model: prompt inspo from Jia et al., 2025
def _build_prompt(text: str, definition: str, tokenizer=None, max_text_tokens: int = 850) -> str:
    if tokenizer is not None:
        text_ids = tokenizer.encode(text, add_special_tokens=False)
        if len(text_ids) > max_text_tokens:
            text = tokenizer.decode(text_ids[:max_text_tokens])

    p_text = f'Turn text: "{text}"'
    p_context = f"Construct: {definition}"
    p_query = "Based only on the text above, does this turn express the construct described?"
    p_output_constraints = "Answer with exactly one word: Yes or No."

    return f"{p_text}\n\n{p_context}\n\n{p_query}\n\n{p_output_constraints}"


def _score_batch(texts: list, definition: str, batch_size: int = 8, max_length: int = 1024) -> list:
    tokenizer, model = _load_zeroshot_model()
    yes_ids, no_ids = _yes_no_token_ids(tokenizer)
    candidate_ids = yes_ids + no_ids

    # Sort by length so each batch has similar-length prompts,
    # less memory-shape variance between batches
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    sorted_texts = [texts[i] for i in order]

    all_scores_sorted = []
    for i in range(0, len(sorted_texts), batch_size):
        chunk = sorted_texts[i:i + batch_size]
        prompts = [
          tokenizer.apply_chat_template(
              [{"role": "user", "content": _build_prompt(t, definition, tokenizer=tokenizer)}],
                add_generation_prompt=True,
                tokenize=False,
            )
            for t in chunk
        ]
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        )
        if torch.cuda.is_available():
            inputs = {k: v.to(model.device) for k, v in inputs.items()}

        attention_mask = inputs["attention_mask"]
        position_ids = attention_mask.cumsum(-1) - 1
        position_ids.masked_fill_(attention_mask == 0, 0)

        with torch.inference_mode():
            out = model(**inputs, position_ids=position_ids)
            next_token_logits = out.logits[:, -1, :]

        # Logprob approach: Li et al., 2025
        probs = F.softmax(next_token_logits[:, candidate_ids], dim=-1)
        p_yes = probs[:, :len(yes_ids)].sum(dim=-1)
        all_scores_sorted.extend(p_yes.tolist())

        del out, next_token_logits, probs

    # restore original order
    all_scores = [None] * len(texts)
    for orig_i, score in zip(order, all_scores_sorted):
        all_scores[orig_i] = score

    return all_scores


def compute_llm_zeroshot_signal(turns_df: pd.DataFrame, signal_params: dict,
                                 signal_name: str, batch_size: int = 8) -> pd.DataFrame:
    definition = signal_params.get("definition", "")
    if not definition:
        raise ValueError(
            f"No 'definition' for '{signal_name}' — add it to signal_params.json's "
            f"entry for this signal's params_key."
        )

    result_df = pd.DataFrame({"score": np.nan}, index=turns_df.index)
    valid_mask = ~turns_df["is_code_like"]
    if not valid_mask.any():
        return result_df

    texts = turns_df.loc[valid_mask, "cleaned_text"].tolist()
    result_df.loc[valid_mask, "score"] = _score_batch(texts, definition, batch_size=batch_size)

    # Try to clear some memory
    torch.cuda.empty_cache()
    gc.collect()

    p_id = turns_df[participant_col].iloc[0] if participant_col in turns_df.columns else "unknown"
    print(f"  [{p_id}] GPU reserved: {torch.cuda.memory_reserved()/1e9:.2f}GB, allocated: {torch.cuda.memory_allocated()/1e9:.2f}GB")

    return result_df

#### All measurement methods

In [ ]:
signal_methods = {
    # "ccr": compute_ccr_signal,
    # "lexicon": compute_lexicon_signal,
    # "sbert": compute_sbert_signal,
    "llm_zeroshot": compute_llm_zeroshot_signal,
}

## Part 3: Run Main Loop on Level 1 Logs


For each `*_level1.csv` in `LEVEL1_LOGS_DIR`, and each **active** row in `signals_config`,
get the right specified measurement method and write one output CSV per (participant x signal) into
`METRICS_DIR`. `participant_id` is read from the CSV's own column rather.

**`clean_and_check()` runs once per participant**, not once per signal. **`clean_and_check()` runs once per participant**, not once per signal. The resulting
`is_code_like`/`cleaned_text` columns are attached to `turns_df` before the signal
loop and reused by every signal's scoring function.

In [ ]:
def slugify(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


PARTICIPANT_FILTER = "P5"  # set to a participant_id string to run just one, or None for all

active_signals = signals_config[signals_config["active"]]
level1_files = sorted(level1_logs_dir.glob("*_level1.csv"))

if not level1_files:
    print(f"No *_level1.csv files found in {level1_logs_dir} — run the preprocessing notebook first.")

for log_path in level1_files:
    turns_df = pd.read_csv(log_path, parse_dates=[timestamp_col])
    participant_id = turns_df[participant_col].iloc[0] if participant_col in turns_df.columns \
        else log_path.stem

    if PARTICIPANT_FILTER and participant_id != PARTICIPANT_FILTER:
        continue

    # Computed once per participant, reused across every signal below.
    cleaned = turns_df[text_col].apply(clean_and_check)
    turns_df["is_code_like"] = cleaned.apply(lambda x: x[0])
    turns_df["cleaned_text"] = cleaned.apply(lambda x: x[1])

    print(f"\n{participant_id}: {len(turns_df)} turns "
          f"({(turns_df[actor_col] == 'user').sum()} user / "
          f"{(turns_df[actor_col] == 'AI').sum()} AI)")
    print(f"  Filter pass rate by actor: "
          f"{(~turns_df['is_code_like']).groupby(turns_df[actor_col]).mean().to_dict()}")

    for _, row in active_signals.iterrows():
        method_name = row["method"]
        signal_name = row["signal"]
        params_key = row["params_key"]

        if method_name not in signal_methods:
            print(f"  [skip] {signal_name} ({method_name}): unknown method — "
                  f"add it to signal_methods first.")
            continue

        score_fn = signal_methods[method_name]
        signal_params = signal_params_by_key.get(params_key, {}) if params_key else {}

        try:
            scores_df = score_fn(turns_df, signal_params, signal_name)
        except Exception as e:
            print(f"  [FAILED] {signal_name} ({method_name}): {type(e).__name__}: {e}")
            continue

        out_df = pd.DataFrame({
            turn_id_col: turns_df[turn_id_col],
            timestamp_col: turns_df[timestamp_col],
            actor_col: turns_df[actor_col],
            word_count_col: turns_df[word_count_col] if word_count_col in turns_df.columns else np.nan,
        })
        out_df = pd.concat([out_df, scores_df], axis=1)

        out_path = metrics_dir / f"{participant_id}_{slugify(signal_name)}_{method_name}.csv"
        out_df.to_csv(out_path, index=False)

        base_cols = {turn_id_col, timestamp_col, actor_col, word_count_col}
        score_cols = [c for c in out_df.columns if c not in base_cols]
        n_scored = out_df[score_cols].notna().any(axis=1).sum()
        print(f"  [ok] {out_path.name}  ({n_scored}/{len(out_df)} turns scored, "
              f"columns: {score_cols})")

In [ ]:
# Upload metrics back to Box
from boxsdk.exception import BoxAPIException

# Upload metrics back to Box
for csv_file in metrics_dir.glob("*.csv"):
    try:
        client.folder(box_metrics_folder_id).upload(str(csv_file))
        print(f"  uploaded (new) {csv_file.name}")
    except BoxAPIException as e:
        if e.code == 'item_name_in_use':
            existing_id = e.context_info['conflicts']['id']
            client.file(existing_id).update_contents(str(csv_file))
            print(f"  updated (overwrote) {csv_file.name}")
        else:
            raise